In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import librosa
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Load the pre-trained VGG16 model for visual feature extraction
vgg_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Function to extract visual features using pre-trained VGG16
def extract_visual_features(video_path, num_segments=5):
    cap = cv2.VideoCapture(video_path)
    frames = []
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (224, 224))
        frames.append(frame)

    cap.release()
    
    # Check if frames are extracted
    if len(frames) == 0:
        print(f"No frames extracted from {video_path}.")
        return None  # Return None if no frames were extracted

    # If the video has fewer frames than segments, pad the frames
    if len(frames) < num_segments:
        for _ in range(num_segments - len(frames)):
            frames.append(np.zeros((224, 224, 3), dtype=np.uint8))

    # Select evenly spaced frames
    frames = np.array(frames)[::max(1, len(frames) // num_segments)]
    
    # Extract features
    features = vgg_model.predict(frames)
    return features.flatten()

# Function to extract audio features (MFCCs)
def extract_audio_features(video_path):
    audio, sr = librosa.load(video_path, sr=None)
    if audio.size == 0:
        print(f"No audio extracted from {video_path}.")
        return None  # Return None if no audio was extracted

    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)
    mean_mfcc = np.mean(mfcc, axis=1)
    std_mfcc = np.std(mfcc, axis=1)
    return np.concatenate([mean_mfcc, std_mfcc])

# Function to extract movement features (optical flow)
def extract_movement_features(video_path, num_segments=5):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    
    if not ret:
        print(f"No frames for movement extraction from {video_path}.")
        return np.zeros(2)  # Return zeros if no frames available

    prev_frame = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    movement_magnitudes = []
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        curr_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        flow = cv2.calcOpticalFlowFarneback(prev_frame, curr_frame, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        magnitude, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        movement_magnitudes.append(np.mean(magnitude))
        
        prev_frame = curr_frame

    cap.release()
    
    return np.array([np.mean(movement_magnitudes), np.std(movement_magnitudes)])

In [2]:

# Path to the folder containing videos and the Excel file with labels
video_folder = "/Users/jeremiah/video-p-reaction-joketype-modular/data/videos" # Replace with your video folder path
excel_file = "/Users/jeremiah/video-p-reaction-joketype-modular/data/excel.xlsx"  # Replace with your Excel file path

In [3]:
# Read the Excel file
df_labels = pd.read_excel(excel_file)

In [4]:
df_labels.columns

Index(['Unnamed: 0', 'VideoID', 'Consent', 'Sharing', 'ParentID', 'ChildID',
       'Age', 'Gender', 'PregnancyLength', 'Languages', 'JokeOrder',
       'ParentFeedback', 'UseOfMedia', 'Withdrawal', 'DatabraryShare',
       'JokeType', 'JokeNum', 'RepNum', 'HowFunny', 'Laugh', 'take',
       'shortname'],
      dtype='object')

In [5]:
# Read the Excel file
df_labels = pd.read_excel(excel_file)

# Prepare the feature list and labels
features = []
labels = []

# Iterate through each video ID in the DataFrame
for index, row in df_labels.iterrows():
    video_id = row['VideoID']  # Adjust the column name as per your Excel structure
    label = row['HowFunny']  # Adjust the column name as per your Excel structure
    video_path = os.path.join(video_folder, f"{video_id}.mp4")  # Adjust the file format if necessary
    
    print(f"Processing {video_path}...")  # Debug statement
    if os.path.exists(video_path):
        # Extract features
        visual_features = extract_visual_features(video_path)
        if visual_features is None:
            print(f"Skipping {video_path} due to visual feature extraction failure.")
            continue  # Skip if visual features extraction failed
            
        audio_features = extract_audio_features(video_path)
        if audio_features is None:
            print(f"Skipping {video_path} due to audio feature extraction failure.")
            continue  # Skip if audio features extraction failed
            
        movement_features = extract_movement_features(video_path)

        # Combine features into a single feature vector
        combined_features = np.concatenate([visual_features, audio_features, movement_features])
        features.append(combined_features)
        labels.append(label)
    else:
        print(f"Video file not found: {video_path}")

# Convert to DataFrame
if features:  # Check if features list is not empty
    feature_df = pd.DataFrame(features)
else:
    print("No features were extracted. Exiting.")
    exit()  # Exit if no features were extracted

label_array = np.array(labels)

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(feature_df)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, label_array, test_size=0.3, random_state=42)

# Build a simple feedforward neural network for classification
model = Sequential()
model.add(Dense(128, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))  # Use 'softmax' for multi-class classification

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=20, batch_size=16, validation_split=0.2)

# Evaluate the model
y_pred = (model.predict(X_test) > 0.5).astype("int32")  # Convert probabilities to binary predictions
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%") 

Processing /Users/jeremiah/video-p-reaction-joketype-modular/data/videos/NFCK3A.joke1.rep1.take1.ThatsNotAHat.mp4.mp4...
Video file not found: /Users/jeremiah/video-p-reaction-joketype-modular/data/videos/NFCK3A.joke1.rep1.take1.ThatsNotAHat.mp4.mp4
Processing /Users/jeremiah/video-p-reaction-joketype-modular/data/videos/NFCK3A.joke1.rep1.take2.ThatsNotAHat.mp4.mp4...
Video file not found: /Users/jeremiah/video-p-reaction-joketype-modular/data/videos/NFCK3A.joke1.rep1.take2.ThatsNotAHat.mp4.mp4
Processing /Users/jeremiah/video-p-reaction-joketype-modular/data/videos/NFCK3A.joke1.rep2.take1.ThatsNotAHat.mp4.mp4...
Video file not found: /Users/jeremiah/video-p-reaction-joketype-modular/data/videos/NFCK3A.joke1.rep2.take1.ThatsNotAHat.mp4.mp4
Processing /Users/jeremiah/video-p-reaction-joketype-modular/data/videos/NFCK3A.joke1.rep3.take1.ThatsNotAHat.mp4.mp4...
Video file not found: /Users/jeremiah/video-p-reaction-joketype-modular/data/videos/NFCK3A.joke1.rep3.take1.ThatsNotAHat.mp4.mp4


NameError: name 'feature_df' is not defined

: 

In [ ]:
import numpy as np
import tensorly as tl
from tensorly.decomposition import tucker

# Assume X1, X2, X3 are your different modalities
X1 = np.random.rand(10, 20, 30)  # Movement data
X2 = np.random.rand(10, 20, 30)  # Speech data
X3 = np.random.rand(10, 20, 30)  # Facial expression data

# Stack them into a 4D tensor
X = np.stack([X1, X2, X3], axis=-1)

# Perform HOSVD
core, factors = tucker(X, rank=[8, 10, 15, 2])

# The core tensor is your fused representation
X_fused = core

# You can project back to the original space if needed
X_reconstructed = tl.tucker_to_tensor((core, factors))

In [ ]:
X

array([[[[4.07314856e-02, 9.06304408e-01, 1.96469314e-01],
         [7.79880089e-01, 4.61018706e-01, 2.77688332e-01],
         [2.89418111e-01, 1.24409148e-01, 2.47085323e-01],
         ...,
         [9.81837132e-01, 4.93097686e-01, 9.93704768e-01],
         [9.45016110e-01, 8.80856331e-02, 3.80220400e-01],
         [2.51434747e-01, 1.85153922e-01, 5.87402431e-01]],

        [[7.24135485e-01, 1.47847383e-01, 7.01445697e-01],
         [4.51278078e-01, 4.23250695e-01, 3.96436459e-01],
         [4.07081472e-01, 7.16339578e-01, 6.11172153e-01],
         ...,
         [4.06894869e-01, 3.20863749e-01, 6.67678396e-01],
         [3.47292895e-01, 2.05369227e-01, 6.18791090e-01],
         [9.33650790e-01, 7.98297379e-01, 8.22567076e-01]],

        [[4.95707592e-01, 7.73099550e-01, 3.24767592e-01],
         [4.73684654e-01, 1.12365868e-01, 9.90051030e-01],
         [6.83323089e-01, 5.88642127e-02, 8.97653203e-01],
         ...,
         [8.62457462e-02, 2.35624118e-01, 6.21945204e-02],
         [

In [ ]:
X_fused

array([[[[ 6.72969936e+01,  3.82242830e-02],
         [ 1.57690584e-02, -3.62784838e-01],
         [-3.22740552e-02, -2.40337511e-02],
         ...,
         [-4.27772432e-04,  1.56119193e-01],
         [ 7.42619804e-03,  1.21496047e-01],
         [ 1.53580865e-02,  1.45440235e-01]],

        [[ 1.67689269e-02,  8.29380965e-01],
         [ 1.43151708e-01, -1.59575478e-01],
         [-8.13411247e-01,  2.16751613e-01],
         ...,
         [-5.58972846e-01, -1.33530329e-01],
         [-1.18303276e-03, -4.28377600e-01],
         [ 1.70325100e-01,  6.45567155e-03]],

        [[-3.46319514e-02, -1.59398370e-01],
         [ 2.25043774e-01, -2.58381333e-01],
         [-5.50108755e-02, -4.14715921e-01],
         ...,
         [-2.63442845e-01,  1.59971300e-02],
         [ 9.02287399e-02, -4.42238050e-01],
         [ 2.61045256e-01,  5.70136830e-01]],

        ...,

        [[ 4.87249295e-02,  2.46103193e-01],
         [ 3.69367658e-02,  1.09991638e+00],
         [ 3.97686352e-02,  1.84776999